# ResNet-18 on CIFAR-10 — activations in memory

`map()` with no path keeps activations in RAM. Downloads on first run:
CIFAR-10 test split (~20 MB) and ResNet-18 weights (~45 MB).

In [8]:
from torchvision.models import ResNet18_Weights, resnet18

from nnact import ActivationMapper
from nnact.utils import Cifar10Samples, activation_loader

dataset = Cifar10Samples(n=512)
print(
    f"{len(dataset)} samples | first id {dataset[0]['id']!r} | "
    f"{tuple(dataset[0]['x'].shape)}"
)


512 samples | first id 'cifar_00000_cat' | (3, 224, 224)


In [15]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
mapper = ActivationMapper(model)

# Which layers can be hooked. depth=2 descends into the blocks; an unknown
# name raises from map() before the first forward pass, suggesting near misses.
mapper.summary(depth=1)

,module,parameters
layer,,
conv1,Conv2d,9408
bn1,BatchNorm2d,128
relu,ReLU,0
maxpool,MaxPool2d,0
layer1,Sequential,147968
layer2,Sequential,525568
layer3,Sequential,2099712
layer4,Sequential,8393728
avgpool,AdaptiveAvgPool2d,0


In [17]:
# The shared helper owns batching; the mapper receives its completed batches.
loader = activation_loader(dataset, batch_size=64)
store = mapper.map(loader, ["layer3", "layer4", "avgpool", "fc"])
store.metadata  # every store records how it was produced

ResNet activations[1/8]  12%|#2         [00:00<?]

RunMetadata(
    model      = ResNet
    parameters = 11.7M
    layers     = layer3, layer4, avgpool, fc
    samples    = 512
    batch_size = 64
    device     = cpu
    seconds    = 2.81s
    throughput = 182/s
    created    = 2026-09-14T19:18:51+00:00
)

In [11]:
# Per-sample shape and total bytes per layer. layer3 is ~100x avgpool.
store.summary()

,shape,elements,bytes
layer,,,
layer3,"(256, 14, 14)",50176,102760448
layer4,"(512, 7, 7)",25088,51380224
avgpool,"(512, 1, 1)",512,1048576
fc,"(1000,)",1000,2048000


In [12]:
# A store is a Dataset too: store[i] is one sample across every layer,
# ordered to match layer_names, with position i matching sample_ids[i].
sample = store[0]
print(store.sample_ids[0])
for act in sample.activations:
    print(f"  {act.layer_name:<10} {tuple(act.tensor.shape)}")

# Whole stacked tensor per layer, for anything downstream.
print("\navgpool stacked:", tuple(store.activations["avgpool"].shape))

cifar_00000_cat
  layer3     (256, 14, 14)
  layer4     (512, 7, 7)
  avgpool    (512, 1, 1)
  fc         (1000,)

avgpool stacked: (512, 512, 1, 1)


In [13]:
# Layer choice dominates storage. One sample is enough to read the shapes
# off summary() and project the cost at any dataset size.
probe_loader = activation_loader(Cifar10Samples(n=1), batch_size=1)
probe = mapper.map(
    probe_loader,
    mapper.available_layers(depth=1),
    progress=False,
).summary()

cost = probe[["shape", "elements"]].copy()
cost["KB / sample"] = probe["elements"] * 4 / 1024
cost["GB / 10k imgs"] = probe["elements"] * 4 * 10_000 / 1024**3
cost.round(2)


,shape,elements,KB / sample,GB / 10k imgs
layer,,,,
conv1,"(64, 112, 112)",802816,3136.00,29.91
bn1,"(64, 112, 112)",802816,3136.00,29.91
relu,"(64, 112, 112)",802816,3136.00,29.91
maxpool,"(64, 56, 56)",200704,784.00,7.48
layer1,"(64, 56, 56)",200704,784.00,7.48
layer2,"(128, 28, 28)",100352,392.00,3.74
layer3,"(256, 14, 14)",50176,196.00,1.87
layer4,"(512, 7, 7)",25088,98.00,0.93
avgpool,"(512, 1, 1)",512,2.00,0.02
